# 🔧 Project FORESIGHT – Notebook 02: Feature Engineering
**NorthBay Living | Demand & Inventory Intelligence**

Demonstrates and validates all 25+ engineered features with visualisations.


In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA = Path('../data/processed')
LAYOUT = dict(paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'))
print('Ready ✓')

In [ ]:
# Load engineered features
feat = pd.read_csv(DATA / 'features_engineered.csv', parse_dates=['date'], low_memory=False)
print(f'Shape: {feat.shape}')
feat.dtypes.value_counts()

## 1. Feature Groups Overview

In [ ]:
feature_groups = {
    'Temporal':     [c for c in feat.columns if any(k in c for k in ['year','month','quarter','week','day','weekend','holiday','season','christmas','sin_','cos_'])],
    'Lag':          [c for c in feat.columns if 'lag' in c],
    'Rolling':      [c for c in feat.columns if 'roll' in c or 'ewma' in c],
    'Price/Promo':  [c for c in feat.columns if any(k in c for k in ['price','promo','promotion'])],
    'Inventory':    [c for c in feat.columns if any(k in c for k in ['inventory','coverage','reorder','eoq','turnover','supply'])],
    'Momentum':     [c for c in feat.columns if any(k in c for k in ['momentum','volatility','trend','acceleration'])],
    'Revenue':      [c for c in feat.columns if 'revenue' in c],
    'Lifecycle':    [c for c in feat.columns if any(k in c for k in ['launch','new_product','mature','cumulative'])],
    'Encoding':     [c for c in feat.columns if any(k in c for k in ['encoded','_enc'])],
}

for group, feats in feature_groups.items():
    print(f'{group:15s} ({len(feats):2d}): {feats[:5]}')

## 2. Correlation with Target (quantity)

In [ ]:
num_cols = feat.select_dtypes(include=[np.number]).columns.tolist()
if 'quantity' in num_cols:
    corr = feat[num_cols].corr()['quantity'].abs().drop('quantity').nlargest(20)
    fig = go.Figure(go.Bar(x=corr.values, y=corr.index, orientation='h',
        marker=dict(color=corr.values, colorscale=[[0,'#6366f1'],[1,'#06b6d4']]),
        text=corr.round(3).values, textposition='outside', textfont_color='white'))
    fig.update_layout(title='Top 20 Features – Correlation with Demand (|r|)', height=550, **LAYOUT)
    fig.show()

## 3. Lag Feature Analysis

In [ ]:
lag_cols = [c for c in feat.columns if c.startswith('quantity_lag')]
sample_sku = feat['stock_code'].value_counts().index[0]
sku_df = feat[feat['stock_code'] == sample_sku].sort_values('date').head(90)

fig = make_subplots(rows=2, cols=3,
    subplot_titles=[f'Lag {l}' for l in [1,7,14,21,28,35]])
lag_days = [1,7,14,21,28,35]
for i, lag in enumerate(lag_days):
    r, c = divmod(i, 3)
    col_name = f'quantity_lag{lag}'
    if col_name in sku_df.columns:
        valid = sku_df.dropna(subset=[col_name])
        fig.add_trace(go.Scatter(
            x=valid['quantity'], y=valid[col_name],
            mode='markers',
            marker=dict(color='#6366f1', size=5, opacity=0.7),
            name=f'lag{lag}'
        ), row=r+1, col=c+1)

fig.update_layout(title=f'Lag Features vs Actual Demand – SKU: {sample_sku}',
    height=500, showlegend=False, **LAYOUT)
fig.show()

## 4. Rolling Statistics

In [ ]:
sku_df = feat[feat['stock_code'] == sample_sku].sort_values('date').head(120)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sku_df['date'], y=sku_df['quantity'],
    name='Actual', line=dict(color='#94a3b8', width=1.5)))
for window, color in [(7,'#6366f1'),(14,'#06b6d4'),(28,'#f59e0b'),(56,'#10b981')]:
    col = f'quantity_roll_mean_{window}'
    if col in sku_df.columns:
        fig.add_trace(go.Scatter(x=sku_df['date'], y=sku_df[col],
            name=f'Roll Mean {window}d', line=dict(color=color, width=2)))
fig.update_layout(title=f'Rolling Mean Features – SKU: {sample_sku}', height=400, **LAYOUT)
fig.show()

## 5. Demand Momentum & Volatility

In [ ]:
if 'demand_momentum' in feat.columns and 'demand_volatility' in feat.columns:
    latest = feat.sort_values('date').groupby('stock_code').last().reset_index()
    fig = px.scatter(latest, x='demand_momentum', y='demand_volatility',
        color='abc_class' if 'abc_class' in latest.columns else None,
        size='quantity' if 'quantity' in latest.columns else None,
        hover_name='stock_code',
        color_discrete_map={'A':'#6366f1','B':'#06b6d4','C':'#f59e0b'},
        title='Demand Momentum vs Volatility by SKU')
    fig.add_hline(y=1.5, line_dash='dash', line_color='#ef4444',
        annotation_text='High Volatility Threshold')
    fig.add_vline(x=1.0, line_dash='dash', line_color='#94a3b8',
        annotation_text='Neutral Momentum')
    fig.update_layout(**LAYOUT, height=450)
    fig.show()

## 6. Feature Statistics Summary

In [ ]:
num_feats = feat.select_dtypes(include=[np.number])
desc = num_feats.describe().T
desc['null_pct'] = (feat.isnull().sum() / len(feat) * 100).round(2)
print(f'Total features: {len(desc)}')
print(f'Features with nulls: {(desc["null_pct"] > 0).sum()}')
print('\nTop 10 features by std:')
print(desc.nlargest(10, 'std')[['mean','std','min','max','null_pct']].to_string())